In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, expr, sum as _sum

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

# 1. Leemos las tablas base que ya tienes
df_presupuestos       = spark.table("workspace.default.presupuestos")
df_detalle            = spark.table("workspace.default.presupuesto_detalle")
df_clientes           = spark.table("workspace.default.clientes")
df_servicios          = spark.table("workspace.default.servicios")
df_pagos              = spark.table("workspace.default.pago_proveedores")
df_factura_cabecera   = spark.table("workspace.default.factura_cabecera")
df_factura_detalle    = spark.table("workspace.default.factura_detalle")
df_proveedores        = spark.table("workspace.default.proveedores")
df_usuarios           = spark.table("workspace.default.usuarios")


In [0]:

# ============================================================
# 2. Construcción de DIMENSIONES
# ============================================================

# -----------------------------
# 2.1 Dimensión Cliente
# -----------------------------
dim_cliente = df_clientes.select(
    col("IdCliente").alias("ClienteKey"),
    "Ruc",
    "RazonSocial",
    "Telefono",
    "Direccion",
    "Distrito",
    "Provincia",
    "Departamento",
    "Rubro",
    "TipoCliente"
).dropDuplicates()

dim_cliente.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_cliente"
)

# -----------------------------
# 2.2 Dimensión Servicio
# -----------------------------
dim_servicio = df_servicios.select(
    col("IdServicio").alias("ServicioKey"),
    col("IdRubro").alias("RubroKey"),
    col("Servicio").alias("NombreServicio"),
    col("Detalle").alias("DetalleServicio")
).dropDuplicates()

dim_servicio.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_servicio"
)

# -----------------------------
# 2.3 Dimensión Proveedor
# -----------------------------
dim_proveedor = df_proveedores.select(
    col("IdProv").alias("ProveedorKey"),
    "Ruc",
    "RazonSocial",
    "Telefono",
    "Direccion",
    "TipoComprobante",
    "Servicio"
).dropDuplicates()

dim_proveedor.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_proveedor"
)

# -----------------------------
# 2.4 Dimensión Fecha
#    (basada en FechaCrea de presupuestos)
# -----------------------------
dim_fecha = df_presupuestos.select(
    col("FechaCrea").alias("FechaKey")
).dropna().dropDuplicates() \
 .select(
    col("FechaKey"),
    year("FechaKey").alias("Anio"),
    month("FechaKey").alias("Mes"),
    dayofmonth("FechaKey").alias("Dia")
)

dim_fecha.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_fecha"
)

In [0]:
from pyspark.sql.functions import (
    col, expr, lit,
    coalesce,
    sum as _sum
)
# ============================================================
# 3. Construcción de la TABLA DE HECHOS
#    fact_evento_financiero
# ============================================================

# -----------------------------
# 3.1 Totales desde presupuesto_detalle
#     (TotalDetalle por IdPresupuesto)
# -----------------------------
detalle_agg = (
    df_detalle
      .groupBy("IdPresupuesto")
      .agg(_sum("SubTotal").alias("TotalDetalle"))
)

# -----------------------------
# 3.2 Monto pagado a proveedores por presupuesto
# -----------------------------
pagos_agg = (
    df_pagos
      .groupBy("IdPresupuesto")
      .agg(_sum("Monto").alias("MontoPagadoProveedores"))
)

# -----------------------------
# 3.3 Integración con presupuestos
#     y cálculo de métricas financieras
# -----------------------------
fact_base = (df_presupuestos
    .join(detalle_agg, on="IdPresupuesto", how="left")
    .join(pagos_agg, on="IdPresupuesto", how="left")
)

fact_evento = (fact_base
    .withColumn(
        "ImportePresupuestado",
        coalesce(col("TotalDetalle"), col("Total"))
    )
    .withColumn(
        "MontoPagado",
        coalesce(col("MontoPagadoProveedores"), lit(0.0))
    )
    .withColumn(
        "MargenEstimado",
        expr("ImportePresupuestado - MontoPagado")
    )
    .withColumn(
        "PorcentajePagado",
        expr(
            "CASE WHEN ImportePresupuestado <> 0 "
            "THEN MontoPagado / ImportePresupuestado "
            "ELSE 0 END"
        )
    )
    .select(
        col("IdPresupuesto").alias("PresupuestoKey"),
        col("IdCliente").alias("ClienteKey"),
        col("FechaCrea").alias("FechaKey"),
        "ImportePresupuestado",
        "MontoPagado",
        "MargenEstimado",
        "PorcentajePagado"
    )
)

# -----------------------------
# 3.4 Persistencia de la tabla de hechos
# -----------------------------
fact_evento.write.mode("overwrite").saveAsTable(
    "workspace.default.fact_evento_financiero"
)